# 🤖 Agente Inteligente para Priorização de Leads — Groq

Aluno: Carlos Ernesto Martins Vieira Neto

Matrícula: 202302530


Este notebook analisa uma base de leads e cria uma lista priorizada para venda ou troca de sistemas ERP. O agente usa um modelo de linguagem hospedado no **Groq**, avalia cinco critérios comerciais e gera uma justificativa individual para cada empresa.

**Pesos utilizados**

- Interesse declarado: **30%**
- ERP atual: **25%**
- Porte da empresa: **20%**
- Faturamento estimado: **15%**
- Origem do lead: **10%**

**Saída principal:** `score - empresa - justificativa - contato - telefone`

In [11]:
# Instalação das bibliotecas necessárias
!pip -q install -U groq openpyxl "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 89.5 MB/s eta 0:00:00


In [2]:
# Bibliotecas e configurações gerais
import io
import json
import os
import time
from getpass import getpass

import pandas as pd
from groq import Groq
from IPython.display import display

MODEL = "openai/gpt-oss-20b"
TAMANHO_LOTE = 10

PESOS = {
    "nota_interesse": 0.30,
    "nota_erp": 0.25,
    "nota_porte": 0.20,
    "nota_faturamento": 0.15,
    "nota_origem": 0.10,
}

print(f"Modelo selecionado: {MODEL}")

Modelo selecionado: openai/gpt-oss-20b


In [3]:
# Recupera a chave dos Secrets do Colab; se não existir, solicita sem exibi-la
try:
    from google.colab import userdata
    api_key = userdata.get("GROQ_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Cole sua GROQ_API_KEY (ela não será exibida): ")

if not api_key or not api_key.startswith("gsk_"):
    raise ValueError("Chave do Groq ausente ou com formato inválido.")

client = Groq(api_key=api_key)
print("✅ Cliente Groq configurado com segurança.")

✅ Cliente Groq configurado com segurança.


## 2. Envie o dataset

Ao executar a próxima célula, selecione o arquivo **`02. leads.csv`** enviado com a atividade.

In [4]:
# Upload do CSV no Google Colab
from google.colab import files

arquivos = files.upload()
nomes_csv = [nome for nome in arquivos if nome.lower().endswith(".csv")]

if not nomes_csv:
    raise ValueError("Nenhum arquivo CSV foi enviado. Execute a célula novamente.")

nome_csv = nomes_csv[0]
df = pd.read_csv(io.BytesIO(arquivos[nome_csv]))
df.columns = [col.strip() for col in df.columns]

print(f"✅ Arquivo carregado: {nome_csv}")
print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")
display(df.head())

Saving 02. leads.csv to 02. leads.csv
✅ Arquivo carregado: 02. leads.csv
Linhas: 10 | Colunas: 12


,id_lead,empresa,segmento,Contato,Fone,cidade,uf,qtd_funcionarios,faturamento_estimado,erp_atual,interesse,origem_lead
0,1,AgroCampo,Varejo,Carlos Mendes,(62) 99124-3801,Anápolis,GO,215,médio,Planilhas,alto,outbound
1,2,MecSul,Distribuição,Rafael Oliveira,(62) 99317-4420,Goiânia,GO,36,baixo,Sem ERP,baixo,evento
2,3,MetalCenter,Varejo,Juliano Ferreira,(34) 99256-7813,Uberlândia,MG,100,alto,ERP simples,alto,inbound
3,4,LogExpress,Construção,André Souza,(34) 99183-2297,Uberlândia,MG,170,médio,Sem ERP,alto,site
4,5,AgroSul,Indústria,Marcos Pereira,(63) 99271-5542,Palmas,TO,241,baixo,ERP simples,baixo,inbound


In [5]:
# Validação da estrutura esperada
COLUNAS_OBRIGATORIAS = {
    "id_lead", "empresa", "Contato", "Fone",
    "qtd_funcionarios", "faturamento_estimado",
    "erp_atual", "interesse", "origem_lead"
}

faltantes = sorted(COLUNAS_OBRIGATORIAS - set(df.columns))
if faltantes:
    raise ValueError(f"O CSV não possui as colunas obrigatórias: {faltantes}")

df["id_lead"] = pd.to_numeric(df["id_lead"], errors="raise").astype(int)
df["qtd_funcionarios"] = pd.to_numeric(df["qtd_funcionarios"], errors="coerce").fillna(0).astype(int)

if df["id_lead"].duplicated().any():
    duplicados = df.loc[df["id_lead"].duplicated(), "id_lead"].tolist()
    raise ValueError(f"Existem IDs de lead duplicados: {duplicados}")

print("✅ Estrutura do dataset validada.")

✅ Estrutura do dataset validada.


## 3. Definição do agente

O modelo atribui uma nota de 0 a 100 para cada critério. O cálculo ponderado final é feito pelo próprio notebook, o que evita erros de soma e mantém os pesos exatamente como definidos na atividade.

In [6]:
SYSTEM_PROMPT = """
Você é um especialista sênior em vendas consultivas B2B de sistemas ERP no Brasil.
Sua tarefa é avaliar empresas com potencial para aquisição ou troca de ERP.

Para cada lead, atribua notas inteiras de 0 a 100, avaliando individualmente:
1. Interesse declarado: alto deve superar médio, que deve superar baixo.
2. ERP atual: ausência de ERP, planilhas ou ERP antigo geralmente indicam maior oportunidade; ERP consolidado reduz a propensão de troca.
3. Porte: considere a quantidade de funcionários, a complexidade operacional e a capacidade de adoção.
4. Faturamento: alto indica maior capacidade de investimento; médio e baixo devem receber notas proporcionais.
5. Origem: indicação, inbound e site normalmente demonstram intenção maior do que evento ou outbound.

Seja consistente entre empresas com características semelhantes.
A justificativa deve ser curta, objetiva, comercial e mencionar os fatores mais relevantes.
Não invente informações que não estejam no dataset.
""".strip()

SCHEMA_RESPOSTA = {
    "type": "json_schema",
    "json_schema": {
        "name": "avaliacao_leads",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "leads": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id_lead": {"type": "integer"},
                            "empresa": {"type": "string"},
                            "nota_interesse": {"type": "integer", "minimum": 0, "maximum": 100},
                            "nota_erp": {"type": "integer", "minimum": 0, "maximum": 100},
                            "nota_porte": {"type": "integer", "minimum": 0, "maximum": 100},
                            "nota_faturamento": {"type": "integer", "minimum": 0, "maximum": 100},
                            "nota_origem": {"type": "integer", "minimum": 0, "maximum": 100},
                            "justificativa": {"type": "string"}
                        },
                        "required": [
                            "id_lead", "empresa", "nota_interesse",
                            "nota_erp", "nota_porte",
                            "nota_faturamento", "nota_origem",
                            "justificativa"
                        ],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["leads"],
            "additionalProperties": False
        },
    },
}

def analisar_lote(registros, tentativa_maxima=3):
    mensagem = (
        "Avalie todos os leads abaixo. Retorne exatamente um item para cada id_lead, "
        "sem omitir nem duplicar empresas.\n\n"
        + json.dumps(registros, ensure_ascii=False)
    )

    for tentativa in range(1, tentativa_maxima + 1):
        try:
            resposta = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": mensagem},
                ],
                response_format=SCHEMA_RESPOSTA,
                temperature=0.1,
                reasoning_effort="low",
                max_completion_tokens=4096,
            )
            conteudo = resposta.choices[0].message.content
            return json.loads(conteudo)["leads"]
        except Exception as erro:
            if tentativa == tentativa_maxima:
                raise RuntimeError(f"Falha após {tentativa_maxima} tentativas: {erro}") from erro
            espera = 5 * (2 ** (tentativa - 1))
            print(f"Tentativa {tentativa} falhou. Nova tentativa em {espera}s...")
            time.sleep(espera)

print("✅ Agente configurado.")

✅ Agente configurado.


In [7]:
# Prepara apenas as informações necessárias para a análise
COLUNAS_ANALISE = [
    "id_lead", "empresa", "segmento", "cidade", "uf",
    "qtd_funcionarios", "faturamento_estimado",
    "erp_atual", "interesse", "origem_lead"
]

colunas_disponiveis = [col for col in COLUNAS_ANALISE if col in df.columns]
dados_para_analise = df[colunas_disponiveis].fillna("").to_dict(orient="records")

print(f"{len(dados_para_analise)} leads preparados em lotes de até {TAMANHO_LOTE}.")

10 leads preparados em lotes de até 10.


## 4. Execução do agente

A próxima célula faz chamadas reais à API do Groq. Para o dataset fornecido, normalmente será feita apenas uma chamada.

In [8]:
avaliacoes = []

for inicio in range(0, len(dados_para_analise), TAMANHO_LOTE):
    lote = dados_para_analise[inicio:inicio + TAMANHO_LOTE]
    numero_lote = inicio // TAMANHO_LOTE + 1
    print(f"Analisando lote {numero_lote} ({len(lote)} leads)...")
    avaliacoes.extend(analisar_lote(lote))

ids_esperados = set(df["id_lead"].tolist())
ids_recebidos = [item["id_lead"] for item in avaliacoes]

if len(ids_recebidos) != len(set(ids_recebidos)):
    raise ValueError("A IA devolveu algum id_lead duplicado. Execute novamente.")
if set(ids_recebidos) != ids_esperados:
    ausentes = sorted(ids_esperados - set(ids_recebidos))
    extras = sorted(set(ids_recebidos) - ids_esperados)
    raise ValueError(f"IDs inconsistentes. Ausentes: {ausentes}; extras: {extras}")

print(f"✅ {len(avaliacoes)} leads avaliados pelo agente.")

Analisando lote 1 (10 leads)...
✅ 10 leads avaliados pelo agente.


In [9]:
# Calcula o score final com os pesos exatos e monta a saída
df_avaliacao = pd.DataFrame(avaliacoes)

for coluna in PESOS:
    df_avaliacao[coluna] = (
        pd.to_numeric(df_avaliacao[coluna], errors="raise")
        .clip(0, 100)
    )

df_avaliacao["score"] = (
    sum(df_avaliacao[coluna] * peso for coluna, peso in PESOS.items())
    .round()
    .astype(int)
)

dados_contato = df[["id_lead", "empresa", "Contato", "Fone"]].copy()
df_detalhado = df_avaliacao.drop(columns=["empresa"]).merge(
    dados_contato, on="id_lead", how="left", validate="one_to_one"
)

df_detalhado = df_detalhado.sort_values(
    by=["score", "empresa"], ascending=[False, True]
).reset_index(drop=True)

df_saida = df_detalhado[[
    "score", "empresa", "justificativa", "Contato", "Fone"
]].copy()

display(
    df_saida.style
    .background_gradient(subset=["score"], cmap="RdYlGn")
    .format({"score": "{:.0f}"})
)

,score,empresa,justificativa,Contato,Fone
0,90,LogExpress,"Interesse alto, sem ERP, porte médio, faturamento médio e origem site.",André Souza,(34) 99183-2297
1,86,MetalCenter,"Interesse alto, ERP simples, porte médio, faturamento alto e origem inbound.",Juliano Ferreira,(34) 99256-7813
2,82,Distribuidora Central,"Interesse alto, ERP antigo, porte médio, faturamento médio e origem site.",Eduardo Martins,(64) 99142-6678
3,79,AgroCampo,"Interesse alto, ERP em planilhas, porte médio, faturamento médio e origem outbound.",Carlos Mendes,(62) 99124-3801
4,76,Distribuidora CentroOeste,"Interesse médio, ERP simples, porte grande, faturamento médio e origem outbound.",Felipe Carvalho,(61) 99388-1204
5,66,Comercial Brasil Norte,"Interesse baixo, ERP simples, porte médio, faturamento alto e origem indicação.",Thiago Alves,(64) 99167-8895
6,61,AgroSul,"Interesse baixo, ERP simples, porte grande, faturamento baixo e origem inbound.",Marcos Pereira,(63) 99271-5542
7,48,MecSul,"Interesse baixo, sem ERP, porte pequeno, faturamento baixo e origem evento.",Rafael Oliveira,(62) 99317-4420
8,43,AgroForte,"Interesse baixo, ERP consolidado, porte pequeno, faturamento médio e origem inbound.",Pedro Assunção,(63) 98745-9087
9,40,LogMaster,"Interesse baixo, ERP simples, porte pequeno, faturamento baixo e origem evento.",Bruno Ribeiro,(62) 99205-7731


## 5. Exportação dos resultados

In [10]:
# Salva a lista principal e a memória detalhada de pontuação
arquivo_csv = "leads_priorizados.csv"
arquivo_excel = "leads_priorizados_detalhado.xlsx"

df_saida.to_csv(arquivo_csv, index=False, encoding="utf-8-sig")
df_detalhado.to_excel(arquivo_excel, index=False)

print(f"✅ Arquivos gerados: {arquivo_csv} e {arquivo_excel}")
files.download(arquivo_csv)
# Se desejar baixar também a memória detalhada, remova o # da linha abaixo:
# files.download(arquivo_excel)

✅ Arquivos gerados: leads_priorizados.csv e leads_priorizados_detalhado.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Resultado

O agente usa o Groq para interpretar o potencial comercial, mas mantém o cálculo final auditável e reproduzível. Para analisar uma nova base compatível, basta executar novamente a partir da etapa de upload.